# 01 — Raw Ingestion

Ingesta de datos crudos a la capa Raw (Delta Lake):
- **Yellow Taxi Trips** (enero 2023, Parquet): ~3M registros
- **Taxi Zone Lookup** (CSV): 263 zonas

Se cargan sin transformacion, validando el schema contra lo esperado en `pipeline_config.py`.

In [ ]:
%run ../config/pipeline_config

In [ ]:
import logging
import time
import urllib.request

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s - %(message)s")
logger = logging.getLogger("raw_ingestion")

start_time = time.time()
metrics = {"stage": "01_raw_ingestion"}

## Descarga con reintentos

In [ ]:
def download_with_retry(url, local_path, max_retries=MAX_RETRIES):
    for attempt in range(max_retries):
        try:
            logger.info(f"Descargando {url} (intento {attempt + 1}/{max_retries})")
            urllib.request.urlretrieve(url, local_path)
            logger.info(f"Descarga completada: {local_path}")
            return
        except Exception as e:
            wait = RETRY_BACKOFF_SECONDS[attempt] if attempt < len(RETRY_BACKOFF_SECONDS) else 60
            logger.warning(f"Error en descarga (intento {attempt + 1}): {e}. Reintentando en {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"No se pudo descargar {url} despues de {max_retries} intentos.")

## Ingesta Yellow Taxi Trips (Parquet)

In [ ]:
download_with_retry(YELLOW_TAXI_URL, YELLOW_TAXI_LOCAL)

df_taxi = spark.read.parquet(YELLOW_TAXI_LOCAL)

# Validar schema contra lo esperado
incoming_cols = set(df_taxi.columns)
expected_cols = set(f.name for f in EXPECTED_TAXI_SCHEMA.fields)
missing_cols = expected_cols - incoming_cols
extra_cols = incoming_cols - expected_cols

if missing_cols:
    logger.error(f"SCHEMA DRIFT: Columnas faltantes: {missing_cols}")
    raise ValueError(f"Schema validation failed: columnas faltantes {missing_cols}")
if extra_cols:
    logger.warning(f"SCHEMA DRIFT: Columnas nuevas no esperadas (se conservan): {extra_cols}")
logger.info(f"Schema OK: {len(expected_cols)} columnas esperadas presentes.")

taxi_count = df_taxi.count()
logger.info(f"Yellow Taxi Trips: {taxi_count:,} registros, {len(df_taxi.columns)} columnas.")

metrics["raw_taxi_records"] = taxi_count
metrics["schema_extra_cols"] = list(extra_cols) if extra_cols else []

In [ ]:
try:
    df_taxi.write.format("delta").mode("overwrite").saveAsTable(RAW_TAXI_TABLE)
    spark.sql(f"COMMENT ON TABLE {RAW_TAXI_TABLE} IS 'Viajes en taxi amarillo NYC - Enero 2023. Fuente: NYC TLC.'")

    props = ", ".join([f"'{k}' = '{v}'" for k, v in TABLE_PROPERTIES["raw"].items()])
    spark.sql(f"ALTER TABLE {RAW_TAXI_TABLE} SET TBLPROPERTIES ({props})")

    logger.info(f"Tabla '{RAW_TAXI_TABLE}' escrita con table properties.")
except Exception as e:
    logger.error(f"Error escribiendo tabla raw taxi: {e}")
    raise

## Ingesta Taxi Zone Lookup (CSV)

In [ ]:
download_with_retry(TAXI_ZONE_URL, TAXI_ZONE_LOCAL)

df_zones = spark.read.option("header", "true").option("inferSchema", "true").csv(TAXI_ZONE_LOCAL)

zones_count = df_zones.count()
logger.info(f"Taxi Zone Lookup: {zones_count} zonas.")

metrics["raw_zones_records"] = zones_count

In [ ]:
try:
    df_zones.write.format("delta").mode("overwrite").saveAsTable(RAW_ZONES_TABLE)
    spark.sql(f"COMMENT ON TABLE {RAW_ZONES_TABLE} IS 'Lookup de zonas TLC (LocationID, Borough, Zone, service_zone).'")

    props = ", ".join([f"'{k}' = '{v}'" for k, v in TABLE_PROPERTIES["raw"].items()])
    spark.sql(f"ALTER TABLE {RAW_ZONES_TABLE} SET TBLPROPERTIES ({props})")

    logger.info(f"Tabla '{RAW_ZONES_TABLE}' escrita con table properties.")
except Exception as e:
    logger.error(f"Error escribiendo tabla raw zones: {e}")
    raise

## Verificacion

In [ ]:
verify_taxi = spark.table(RAW_TAXI_TABLE).count()
verify_zones = spark.table(RAW_ZONES_TABLE).count()

if verify_taxi != taxi_count:
    logger.warning(f"Discrepancia en taxi: esperados {taxi_count}, encontrados {verify_taxi}")
if verify_zones != zones_count:
    logger.warning(f"Discrepancia en zones: esperados {zones_count}, encontrados {verify_zones}")

logger.info(f"Verificacion OK: taxi={verify_taxi:,}, zones={verify_zones}")

spark.table(RAW_TAXI_TABLE).printSchema()
spark.table(RAW_ZONES_TABLE).printSchema()

In [ ]:
elapsed = round(time.time() - start_time, 2)
metrics["duration_seconds"] = elapsed
metrics["status"] = "SUCCESS"

logger.info(f"Raw ingestion completada en {elapsed}s.")
logger.info(f"Metricas: {metrics}")

try:
    import json
    dbutils.notebook.exit(json.dumps(metrics))
except NameError:
    pass